# Returns analysis

Handed over by a placement student who left in July. The last thing they wrote in
the handover email was that the model gets 0.94 and is ready to show the client.

Nobody has run this since.

In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler

In [2]:
df = pd.read_csv('../data/retail_orders_week1.csv')
df.shape

(2030, 17)

In [6]:
# drop the rows that look wrong, and put VAT on the order value
df = pd.read_csv('../data/retail_orders_week1.csv')
df = df[df['quantity'] > 0]
df = df[df['delivery_days'] > 0]
df['order_value'] = df['order_value'] * 1.2
len(df)

2021

In [7]:
sample = df.sample(1200,random_state=7)
sample['returned'].mean()

np.float64(0.1975)

In [8]:
features = ['item_price', 'discount_pct', 'quantity',
            'delivery_days', 'customer_prior_orders']

X = df[features]
y = df['returned']

scaler = StandardScaler()
X = scaler.fit_transform(X)

In [9]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25,random_state=7, stratify=y)

model = LogisticRegression(max_iter=2000)
model.fit(X_train, y_train)

auc = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
print('ROC-AUC', round(auc, 3))

ROC-AUC 0.587


In [10]:
coefs = pd.Series(model.coef_[0], index=features).sort_values()
coefs

customer_prior_orders   -0.019009
delivery_days            0.021380
item_price               0.034294
discount_pct             0.048160
quantity                 0.062325
dtype: float64

In [11]:
summary = (
    df.groupby('category', as_index=False)['returned']
      .mean()
      .rename(columns={'returned': 'return_rate'})
      .sort_values('return_rate', ascending=False)
)
summary

,category,return_rate
3,footwear,0.292683
0,apparel,0.238462
2,electronics,0.203762
5,sports,0.129412
4,home,0.111959
1,beauty,0.079208


In [13]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
import numpy as np

# Start fresh with all 2,030 rows (no filtering)
df_full = pd.read_csv('../data/retail_orders_week1.csv')

# Define numeric and categorical columns
numeric_features = ['export_seq', 'item_price', 'discount_pct', 'quantity',
                    'delivery_days', 'customer_prior_orders']
categorical_features = ['category', 'channel', 'payment_method']

# Test each numeric column individually
results = []

for col in numeric_features:
    X = df_full[[col]]
    y = df_full['returned']

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    model = LogisticRegression(max_iter=2000, random_state=0)

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
    scores = cross_val_score(model, X_scaled, y, cv=cv, scoring='roc_auc')

    mean_auc = scores.mean()
    results.append({'column': col, 'mean_auc': mean_auc, 'std': scores.std()})

    print(f"{col:25} | ROC-AUC: {mean_auc:.4f} ± {scores.std():.4f}")

results_df = pd.DataFrame(results).sort_values('mean_auc', ascending=False)
print("\n" + "="*50)
print(results_df)

export_seq                | ROC-AUC: 0.9335 ± 0.0048
item_price                | ROC-AUC: 0.5282 ± 0.0166
discount_pct              | ROC-AUC: 0.5283 ± 0.0292
quantity                  | ROC-AUC: 0.5214 ± 0.0287
delivery_days             | ROC-AUC: 0.5009 ± 0.0225
customer_prior_orders     | ROC-AUC: 0.4699 ± 0.0208

                  column  mean_auc       std
0             export_seq  0.933471  0.004815
2           discount_pct  0.528291  0.029170
1             item_price  0.528202  0.016644
3               quantity  0.521397  0.028651
4          delivery_days  0.500866  0.022471
5  customer_prior_orders  0.469859  0.020823
